# NAMAA Egyptian voice — Kaggle free GPU smoke test

Uses the official `NAMAA-Space/NAMAA-Egyptian-TTS` model. Runs on Kaggle T4 GPU only. No Project 79 server. Produces `/kaggle/working/namaa_test.wav` and measured audio runtime.


In [ ]:
%pip install -q "chatterbox-tts" "huggingface_hub" "safetensors" "torchaudio"


In [ ]:
import os, time, traceback
from pathlib import Path
print("NAMAA test starting", flush=True)
import torch
print("CUDA_AVAILABLE", torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("Free Kaggle GPU is not active; set Accelerator to GPU T4 x2.")
print("GPU_DEVICE", torch.cuda.get_device_name(0), flush=True)

from huggingface_hub import snapshot_download
from safetensors.torch import load_file as load_safetensors
from chatterbox import mtl_tts
import torchaudio as ta

start=time.monotonic()
print("DOWNLOADING_NAMAA_MODEL", flush=True)
ckpt_dir=snapshot_download(repo_id="NAMAA-Space/NAMAA-Egyptian-TTS",repo_type="model")
print("MODEL_DOWNLOADED", round(time.monotonic()-start,1), flush=True)
model=mtl_tts.ChatterboxMultilingualTTS.from_pretrained(device="cuda")
state=load_safetensors(f"{ckpt_dir}/t3_mtl23ls_v2.safetensors",device="cpu")
model.t3.load_state_dict(state)
model.t3.to("cuda").eval()
print("MODEL_LOADED", round(time.monotonic()-start,1), flush=True)

text="تخيل إن فيه ملك كان معاه دهب أكتر من اللي أي حد ممكن يتصوره! اسمه مانسا موسى."
out="/kaggle/working/namaa_test.wav"
begin=time.monotonic()
with torch.inference_mode():
    wav=model.generate(text,language_id="ar")
ta.save(out,wav.detach().cpu(),model.sr)
print("SUCCESS",out,"BYTES",Path(out).stat().st_size,"SAMPLE_RATE",model.sr,"AUDIO_SECONDS",round(wav.shape[-1]/model.sr,2),"GENERATE_SECONDS",round(time.monotonic()-begin,1),flush=True)
